# 01 — Explore ResPlan

Sanity-checks the data pipeline before any training happens: downloads ResPlan, renders a few real plans, and builds one sample through `floorplan_gen`'s own dataset code so we can see exactly what the model will be trained on.

Run cells top to bottom. See `../README.md` for what this project is (and isn't).

## Setup

Two ways to get the `floorplan_gen` code into this Colab session — pick one:

- **Option A** (once you've pushed this repo to GitHub): set `GITHUB_REPO_URL` below.
- **Option B** (no GitHub push needed): mount Drive and upload the `ml/floor-plan-gen` folder there first, then set `DRIVE_PROJECT_PATH` to match.

In [ ]:
GITHUB_REPO_URL = ""  # e.g. "https://github.com/<you>/planHouse.git"
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/planHouse-ml/floor-plan-gen"  # used if GITHUB_REPO_URL is empty

import os
import sys

if GITHUB_REPO_URL:
    if not os.path.isdir("/content/planHouse"):
        !git clone {GITHUB_REPO_URL} /content/planHouse
    else:
        !git -C /content/planHouse pull  # already cloned earlier this session — just update
    PROJECT_DIR = "/content/planHouse/ml/floor-plan-gen"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = DRIVE_PROJECT_PATH
    assert os.path.isdir(PROJECT_DIR), (
        f"{PROJECT_DIR} not found — upload ml/floor-plan-gen there first, "
        f"or set GITHUB_REPO_URL above instead."
    )

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))
%pip install -q -r {os.path.join(PROJECT_DIR, "requirements.txt")}

## Download the dataset

Idempotent — re-running this cell in a later session skips files that already exist in `data/`.

In [ ]:
from pathlib import Path

from floorplan_gen.data.download import download_resplan, load_split

DATA_DIR = Path(PROJECT_DIR) / "data"
download_resplan(DATA_DIR)
splits = load_split(DATA_DIR)
{k: len(v) for k, v in splits.items()}

## Render a few real plans

Using ResPlan's own `plot_plan` (from the `resplan_utils.py` we just downloaded) — this is the ground truth the model is trained to imitate, not our code.

In [ ]:
import pickle
import sys as _sys

import matplotlib.pyplot as plt

_sys.path.insert(0, str(DATA_DIR))  # so `import resplan_utils` works
import resplan_utils

with open(DATA_DIR / "ResPlan.pkl", "rb") as f:
    plans = pickle.load(f)
print(f"Loaded {len(plans)} plans")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, idx in zip(axes, splits["train"][:4]):
    resplan_utils.plot_plan(plans[idx], ax=ax, legend=False, title=f"plan {idx}")
plt.tight_layout()
plt.show()

## Inspect one sample through our own pipeline

This is exactly what the model sees during training: room types, the typed adjacency graph, a rasterized boundary mask, and normalized target boxes.

In [ ]:
from floorplan_gen.data.resplan_dataset import EDGE_TYPES, ROOM_TYPES, plan_to_pyg_data

sample = plan_to_pyg_data(plans[splits["train"][0]], resplan_utils)

room_type_names = [ROOM_TYPES[t] if t < len(ROOM_TYPES) else "unknown" for t in sample.room_type.tolist()]
print("rooms:", room_type_names)
print("edges:", sample.edge_index.shape[1])
print("edge types used:", sorted({EDGE_TYPES[t] if t < len(EDGE_TYPES) else "unknown" for t in sample.edge_type.tolist()}))
print("boundary_mask shape:", tuple(sample.boundary_mask.shape))
print("target boxes (normalized x, y, w, h):")
print(sample.y)

## End-to-end dataset smoke test

Builds the full `val` split through `ResPlanDataset` — if this cell runs without error, the whole data pipeline (pickle -> graph -> tensors) works, and `02_train_baseline.ipynb` should be safe to run next.

In [ ]:
from floorplan_gen.data.resplan_dataset import ResPlanDataset

val_ds = ResPlanDataset(DATA_DIR, "val")
print(f"val dataset: {len(val_ds)} usable samples")
print(val_ds[0])